# MIND quickstart — toy data, no download

This notebook walks through the entire MIND pipeline on a small synthetic dataset. It runs in well under a minute on a laptop CPU and requires no internet access. Use it to:

- verify the package is installed correctly,
- understand the input-format conventions, and
- see exactly which calls you need to swap in your own data.

If anything below errors, please open an issue at <https://github.com/hwxing3259/MIND/issues>.

## 1. Generate a toy multi-omics dataset

`make_toy_dataset` returns a dictionary in exactly the shape `MIND.MIND` expects: one DataFrame per modality with aligned row indices, plus a `cls` DataFrame with the (latent) cluster label for each patient.

In [ ]:
from MIND.data import make_toy_dataset

data = make_toy_dataset(
    n=200,
    n_modalities=3,
    n_features=50,
    n_clusters=3,
    missing_rate=0.2,
    seed=0,
)
for k, df in data.items():
    print(f'{k:12s} shape={df.shape}  missing rows={df.isna().to_numpy().all(axis=1).sum()}')

## 2. Train MIND

We strip the `cls` label out (it's only used for evaluation) and pass the remaining modalities into `MIND`. A few hundred epochs is enough on this toy data.

In [ ]:
import torch
from MIND import MIND

torch.manual_seed(0)
modalities = {k: v for k, v in data.items() if k != 'cls'}
model = MIND(data_dict=modalities, emb_dim=16, device='cpu')
model.my_train(n_epoch=300, lr=1e-3, verbose=False)

## 3. Inspect the embedding

We project the 16-D embedding to 2-D with PCA and colour by the true cluster label. A successful run shows three roughly-separated blobs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

with torch.no_grad():
    z_mean, _ = model.get_embedding()
z = z_mean.cpu().numpy()

z_centered = z - z.mean(axis=0, keepdims=True)
u, s, vt = np.linalg.svd(z_centered, full_matrices=False)
z2 = z_centered @ vt[:2].T

labels = data['cls']['label'].to_numpy()
fig, ax = plt.subplots(figsize=(5, 5))
for c in np.unique(labels):
    m = labels == c
    ax.scatter(z2[m, 0], z2[m, 1], label=f'cluster {c}', s=20, alpha=0.7)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend()
ax.set_title('MIND embedding (toy data)')
plt.tight_layout()
plt.show()

## 4. Predict missing modalities

`predict()` returns one tensor per modality. Where a row was originally NaN, the output is the model's *prediction* of what those values would have been.

In [ ]:
recons = model.predict()
for name, r, original in zip(modalities.keys(), recons, modalities.values()):
    missing_mask = original.isna().to_numpy().all(axis=1)
    print(f'{name}: imputed {missing_mask.sum()} previously-missing rows  (recon shape={tuple(r.shape)})')

## 5. Swap in your own data

Replace step 1 with your own dictionary of DataFrames. The constraints are documented in [`docs/user_guide.md`](../docs/user_guide.md):

- one row per patient, identical across modalities,
- missing patients encoded as **all-NaN rows** (not partial NaNs), and
- features z-scored per modality.

Everything from step 2 onwards stays the same.